# BenchMark comprasion

In [3]:
import numpy as np 
import pandas as pd 
from sklearn.metrics import accuracy_score,f1_score
import json
sibling_b_path = "logs/sibling_b.csv"
metrics_log = "logs/metrics_log.csv"
output_json = "results/benchmark_results.json"
def load_encode_data(path = sibling_b_path):
    df = pd.read_csv(path)
    print("sibling b data loading is completed")
    all_chars = set()
    for col in df.columns:
        all_chars.update(df[col].astype(str).unique())
    vocab = sorted(all_chars)
    char_to_id = {}
    for i,ch in enumerate(vocab):
        char_to_id[ch] = i
    y_true = df["Readable_Actual_Target"].astype(str).map(char_to_id).values
    y_pred_knn = df["KNN_Prediction"].astype(str).map(char_to_id).values
    y_pred_dt    = df["DT_Prediction"].astype(str).map(char_to_id).values
    y_pred_label = df["LabelProp_Prediction"].astype(str).map(char_to_id).values

    np.save("results/y_true.npy",       y_true)
    np.save("results/y_pred_knn.npy",   y_pred_knn)
    np.save("results/y_pred_dt.npy",    y_pred_dt)
    np.save("results/y_pred_label.npy", y_pred_label)
    return y_true,y_pred_knn,y_pred_dt,y_pred_label
def load_neural_net_loss(csv_path = metrics_log):
    df = pd.read_csv(csv_path)
    final_train_loss = round(float(df["train_loss"].iloc[-1]))
    final_val_loss = round(float(df["val_loss"].iloc[-1]))
    best_val_loss = round(float(df["val_loss"].min()))
    best_step = int(df.loc[df["val_loss"].idxmin(),"step"])
    total_steps = int(df["step"].iloc[-1])
    accuracy =  1.2606
    f1 = 0.2254
    stats = {
        "total_steps" : total_steps ,
        "final_train_loss" : final_train_loss,
        "final_val_loss" : final_val_loss,
        "best_val_loss" : best_val_loss,
        "best_step" : best_step,
    }
    
    print("\n==================================================")
    print("  NEURAL NET — Stats from metrics_log.csv")
    print("==================================================")
    print(f"  Total Training Steps   : {total_steps}")
    print(f"  Final Train Loss       : {final_train_loss}")
    print(f"  Final Val Loss         : {final_val_loss}")
    print(f"  best value loss        : {best_val_loss}")
    print(f"  best_step : {best_step}")
    print(f"  total_steps: {total_steps}")
    print(" Accuracy and f1 score of nerual \n ")
    print(f"PyTorch Neural Net | Accuracy :{accuracy} | F1 score :{f1}\n")
if __name__ == "__main__":
    all_results = []
    neural_net = load_neural_net_loss()
    y_true,y_pred_knn,y_pred_dt,y_pred_label = load_encode_data()
    models = {
        "KNN Classifier (Sibling B)": y_pred_knn,
        "Decision Tree (Sibling B)": y_pred_dt,
        "Label Propagation (Sibling A)": y_pred_label
    }
    print("model evaluation result")
    for name,predictions in models.items():
        acc = accuracy_score(y_true,predictions)
        f1 = f1_score(y_true, predictions, average="macro", zero_division=0)
        print(f"{name} - >Accuracy :{acc:.4f} | F1 score :{f1:.4f}")
        all_results.append({
            "name": name,
            "accuracy": round(float(acc), 4),
            "f1": round(float(f1), 4)
        })
    nn_neural ={
        "name" : "PyTorch Neural Net",
        "accuracy" : 1.2606,
        "f1" : 0.2254
    }
    all_results.append(nn_neural)
    with open(output_json, "w") as f:
        json.dump(all_results, f, indent=2)
    
        

    print(f"saved json file {output_json}")
        


  NEURAL NET — Stats from metrics_log.csv
  Total Training Steps   : 29999
  Final Train Loss       : 1
  Final Val Loss         : 1
  best value loss        : 1
  best_step : 29500
  total_steps: 29999
 Accuracy and f1 score of nerual 
 
PyTorch Neural Net | Accuracy :1.2606 | F1 score :0.2254

sibling b data loading is completed
model evaluation result
KNN Classifier (Sibling B) - >Accuracy :0.2752 | F1 score :0.1132
Decision Tree (Sibling B) - >Accuracy :0.3961 | F1 score :0.2380
Label Propagation (Sibling A) - >Accuracy :0.4048 | F1 score :0.2662
saved json file results/benchmark_results.json


# plotting the graph

In [6]:
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
def plot_training(csv_path = "logs/metrics_log.csv"):
    df = pd.read_csv(csv_path)
    plt.figure(figsize = (10,5))
    plt.plot(df["step"],df["train_loss"],label = "Train Loss", color ="blue")
    plt.plot(df["step"], df["val_loss"], label="Validation Loss", color="orange")
    plt.title("Neural Network Training Loss")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True , alpha = 0.3)

    plt.savefig("results/training_loss.png", dpi=150)
    print("graph is saved successful")
    plt.close()
def plot_accuracy(sibling_path = "logs/sibling_b.csv"):
    df = pd.read_csv(sibling_path)
    y_true = df["Readable_Actual_Target"]
    knn_acc = accuracy_score(y_true, df["KNN_Prediction"])
    dt_acc = accuracy_score(y_true, df["DT_Prediction"])
    label_acc = accuracy_score(y_true , df["LabelProp_Prediction"])
    models = ["KNN", "Decision Tree", "Label Propagation"]
    scores = [knn_acc, dt_acc, label_acc]

    plt.figure(figsize=(8, 5))
    plt.bar(models, scores, color=["steelblue", "coral", "mediumseagreen"])
    plt.title("Model Accuracy")
    plt.ylabel("Accuracy")
    plt.ylim(0,1.0)
    plt.savefig("results/overall_accuracy.png", dpi=150)
    print("overall accuracy is saved succesfully")
    plt.close()
if __name__ == "__main__":
    print("Plotting Graph")
    plot_training()
    plot_accuracy()
    print("Done")

Plotting Graph
graph is saved successful
overall accuracy is saved succesfully
Done


# Performance forecaster

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
def load_training(metrics_path="logs/metrics_log.csv"):
    df = pd.read_csv(metrics_path)
    x = df["step"].values.reshape(-1,1)
    y = df["val_loss"].values
    print(f"Loaded {len(df)} training records")
    return x,y
if __name__ == "__main__":
    print("Performance Forecaster")
    x_train,y_train = load_training()
    if x_train is not None:

        print("Training the Regression Model")
        model_linear = LinearRegression()
        model_linear.fit(x_train,y_train)

        decision_tree = DecisionTreeRegressor(max_depth =3 )
        decision_tree.fit(x_train,y_train)
        
        future_steps = np.array([40000, 50000]).reshape(-1, 1)
        linear_prediction = model.predict(future_steps)
        decisiontree_prediction = model.predict(future_steps)
        print("Forecasted val loss for further steps")
        print(f"  {'Future Steps':<15} | {'Linear Regression':<17} | {'Decision Tree'}")
        print("-" * 55)
        
        for step, lin_pred, tree_pred in zip(future_steps, linear_prediction, decisiontree_prediction):
            print(f"  {step[0]:<15,} | {lin_pred:<17.4f} | {tree_pred:.4f}")
        print("Done")

Performance Forecaster
Loaded 121 training records
Training the Regression Model
Forecasted val loss for further steps
  Future Steps    | Linear Regression | Decision Tree
-------------------------------------------------------
  40,000          | 1.2583            | 1.2583
  50,000          | 1.2499            | 1.2499
Done
